# RNA3D — Phase 2: Official PDB Release Dates & Historical RNA Sequence Candidate Screen (CPU)

**Scope:** Existing frozen pilot-10 FASTAs and provenance audit; no model inference, no GPU, no original-file overwrite, no GitHub changes.

**Research boundaries:** This notebook checks official PDB release dates via RCSB's Data API and searches the current PDB archive's RNA polymer entities for *candidate* sequence homologs using the RCSB Search API. Retrieved hits are subsequently filtered by **entry release date**. Search results are **not a historical snapshot** and are **not a complete check of model training data or 3D template similarity**. 'No candidate' is never called 'leak-free' or 'template-sparse'. Checkpoint-specific model data cutoff must be separately authenticated.

**Prerequisite:** Earlier evaluation `pilot10_template_leakage_evidence_inventory_20261113_v1` finalized in Google Drive. This is a new independent evidence layer.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import csv, json, hashlib, re, time, requests, math, os, shutil
from datetime import datetime, timezone
ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
EVIDENCE = ROOT/'Evaluations/pilot10_template_leakage_evidence_inventory_20261113_v1'
FASTA_ROOT = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z/inputs'
AUDIT_ROOT = ROOT/'Evaluations/pilot10_native_chain_provenance_20261113_v1'
OUT_ID = 'pilot10_rcsb_release_historical_sequence_candidates_20261113_v1'
OUTPUT = ROOT/'Evaluations'/OUT_ID
STAGE = Path('/content/rna3d_rcsb_historical_screen_stage')
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
# Cutoffs are screening SCENARIOS, not claims about any exact model checkpoint.
CUTOFFS = {'pre2023': '2022-12-31', 'pre2024': '2023-12-31'}
# Searching an RNA sequence >=25nt: RCSB MMseqs2 documented API.
IDENTITY_CUTOFF = 0.30
EVALUE_CUTOFF = 1.0
MAX_RESULTS_PER_TARGET = 1000
session = requests.Session()
session.headers.update({'User-Agent':'RNA3D-template-scarcity-research/1.0 (academic audit)'})

def sha256(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for part in iter(lambda:f.read(1<<20),b''):h.update(part)
 return h.hexdigest()

def read_csv(p):
 with p.open(newline='') as f:return list(csv.DictReader(f))

def fasta_seq(path):
 lines=[line.strip() for line in path.read_text().splitlines() if line.strip()]
 assert lines[0].startswith('>') and sum(x.startswith('>') for x in lines)==1
 seq=''.join(x for x in lines if not x.startswith('>')).upper()
 assert seq and set(seq)<=set('ACGU') and len(seq)>=25
 return seq

assert len(TARGETS)==len(set(TARGETS))==10
assert EVIDENCE.is_dir() and AUDIT_ROOT.is_dir() and FASTA_ROOT.is_dir()
assert not OUTPUT.exists(),f'Immutable output exists: {OUTPUT}. Inspect it, or set a NEW OUT_ID.'
assert not STAGE.exists(),f'Local staging exists: {STAGE}. Inspect before rerun.'
print('CPU only. No inference. Output:',OUTPUT)


In [ ]:
prior = json.loads((EVIDENCE/'FINALIZED.json').read_text())
assert prior['status']=='complete'
for name,digest in prior['files'].items():
 assert sha256(EVIDENCE/name)==digest, f'Evidence inventory checksum drift: {name}'
rows = read_csv(EVIDENCE/'target_evidence_inventory.csv')
assert len(rows)==10 and set(x['target_id'] for x in rows)==set(TARGETS)
prior_audit=json.loads((AUDIT_ROOT/'FINALIZED.json').read_text())
assert prior_audit['status']=='complete'
for name,digest in prior_audit['files'].items():
 assert sha256(AUDIT_ROOT/name)==digest, f'Native audit checksum drift: {name}'
seq_by_target={t:fasta_seq(FASTA_ROOT/t/f'{t}.fasta') for t in TARGETS}
for r in rows:assert int(r['fasta_length'])==len(seq_by_target[r['target_id']])
print('SOURCE INTEGRITY PASS: 10/10 target FASTAs + immutable previous audits')


In [ ]:
DATA_API='https://data.rcsb.org/rest/v1/core/entry/'
SEARCH_API='https://search.rcsb.org/rcsbsearch/v2/query'
metadata_cache={}
request_log=[]
def get_entry(pdb_id):
 key=pdb_id.upper()
 if key not in metadata_cache:
  u=DATA_API+key
  resp=session.get(u,timeout=35)
  if resp.status_code != 200:raise RuntimeError(f'RCSB Data API {key}: HTTP {resp.status_code}: {resp.text[:250]}')
  doc=resp.json()
  info=doc.get('rcsb_accession_info') or {}
  date=(info.get('initial_release_date') or '').split('T')[0]
  if not re.fullmatch(r'\d{4}-\d{2}-\d{2}',date):raise ValueError(f'RCSB release date unavailable {key}')
  metadata_cache[key]={'pdb_id':key,'official_release_date':date,'deposition_date':str(info.get('deposit_date','')).split('T')[0],
    'entry_title':(doc.get('struct') or {}).get('title',''),'source':'RCSB Data API; queried live'}
  request_log.append({'url':u,'http_status':resp.status_code,'fetched_utc':datetime.now(timezone.utc).isoformat()})
  time.sleep(0.08)
 return metadata_cache[key]

release_rows=[]
for row in rows:
 t=row['target_id'];entry=get_entry(row['pdb_id']);release_rows.append({
 'target_id':t,'pdb_id':entry['pdb_id'],'native_audit_status':row['native_audit_status'],
 'inventory_deposition_date':row['deposition_date'],
 'official_initial_release_date':entry['official_release_date'],
 'release_after_2023_cutoff':entry['official_release_date']>'2023-12-31',
 'release_after_2022_cutoff':entry['official_release_date']>'2022-12-31',
 'official_release_source':'RCSB Data API /core/entry',
 'date_does_not_prove_training_exclusion':True})
print('OFFICIAL RCSB RELEASE DATES: 10/10')
for row in release_rows:print(row['target_id'],row['official_initial_release_date'],row['native_audit_status'])


In [ ]:
# RCSB Search API sequence similarity is performed on the PRESENT archive.
# Historical filtering happens AFTER retrieval. Exhaustive historical screening requires
# a time-frozen archived sequence database and explicit model training data.

def rna_search(seq, start=0, rows=MAX_RESULTS_PER_TARGET):
 query={'query':{'type':'terminal','service':'sequence',
   'parameters':{'evalue_cutoff':EVALUE_CUTOFF,'identity_cutoff':IDENTITY_CUTOFF,
                 'sequence_type':'rna','value':seq}},
   'return_type':'polymer_entity',
   'request_options':{'paginate':{'start':start,'rows':rows},'results_verbosity':'compact'}}
 resp=session.post(SEARCH_API,json=query,timeout=120)
 if resp.status_code == 204:
  return [],0
 if resp.status_code != 200:
  raise RuntimeError(f'RCSB Sequence API HTTP {resp.status_code}: {resp.text[:900]}')
 j=resp.json();return [h['identifier'] for h in j.get('result_set',[])],int(j.get('total_count',len(j.get('result_set',[]))))

# Fail loud on query/API errors; NEVER convert errors into zero-hit claims.
pilot_target='8ubd_H'
hits,total=rna_search(seq_by_target[pilot_target],rows=min(100,MAX_RESULTS_PER_TARGET))
print('RCSB RNA SEARCH PILOT:',pilot_target,'received',len(hits),'out of',total,'hits')
print('Example entity IDs:',hits[:8])


In [ ]:
# Search all ten using exactly the same query parameters.
search_state=[]
found_entities={}
for t in TARGETS:
 try:
  entities,total=rna_search(seq_by_target[t],rows=MAX_RESULTS_PER_TARGET)
  # Truncated responses are marked explicitly, not interpreted as exhaustive.
  truncated=total>len(entities)
  found_entities[t]=entities
  search_state.append({'target_id':t,'returned_entities':len(entities),'total_reported':total,
                       'truncated':truncated,'search_status':'PARTIAL' if truncated else 'COMPLETE_QUERY_AT_THRESHOLD'})
  print(t, 'returned',len(entities),'of',total,'truncated?',truncated)
 except Exception as exc:
  search_state.append({'target_id':t,'returned_entities':0,'total_reported':None,
                       'truncated':True,'search_status':'ERROR','error':repr(exc)})
  found_entities[t]=[]
  print(t,'ERROR:',repr(exc))
 time.sleep(0.2)
assert len(search_state)==10
# API errors must be resolved before any finalization to avoid silently missing targets.
errors=[r for r in search_state if r['search_status']=='ERROR']
assert not errors, f'RCSB search errors; inspect and rerun after fixing: {errors}'


In [ ]:
# Resolve hits' PDB initial release dates. These might be many entities from the same entry.
unique_pdbs=sorted({entity.split('_')[0].upper() for es in found_entities.values() for entity in es})
print('Unique PDB entries returned:',len(unique_pdbs))
meta_errors=[]
for i,pdb in enumerate(unique_pdbs,1):
 try:get_entry(pdb)
 except Exception as exc:meta_errors.append((pdb,repr(exc)))
 if i%100==0:print('Checked',i,'/',len(unique_pdbs))
# Metadata lookup failures mean historical candidate counts are unknown, not zero.
print('Metadata errors:',len(meta_errors))
if meta_errors:print(meta_errors[:10])


In [ ]:
date_by_target={r['target_id']:r['official_initial_release_date'] for r in release_rows}
search_by_target={r['target_id']:r for r in search_state}
candidate_rows=[]
summary_rows=[]
for t in TARGETS:
 state=search_by_target[t]
 bad_meta=[e for e in found_entities[t] if e.split('_')[0].upper() not in metadata_cache]
 for entity in found_entities[t]:
  pdb=entity.split('_')[0].upper()
  if pdb not in metadata_cache:continue
  rel=metadata_cache[pdb]['official_release_date']
  candidate_rows.append({'target_id':t,'candidate_entity_id':entity,'candidate_pdb_id':pdb,
        'candidate_initial_release_date':rel,'is_self_pdb':pdb==t.split('_')[0].upper(),
        'before_end_2022':rel<=CUTOFFS['pre2023'],'before_end_2023':rel<=CUTOFFS['pre2024'],
        'candidate_only_not_training_proof':True})
 nonself=[x for x in candidate_rows if x['target_id']==t and not x['is_self_pdb']]
 old2022=sum(x['before_end_2022'] for x in nonself)
 old2023=sum(x['before_end_2023'] for x in nonself)
 incomplete=state['truncated'] or bool(bad_meta)
 summary_rows.append({'target_id':t,'official_target_release_date':date_by_target[t],
   'rna_search_returned_entities':len(found_entities[t]),'rna_search_total_reported':state['total_reported'],
   'search_truncated':state['truncated'],'unresolved_candidate_metadata':len(bad_meta),
   'nonself_candidates_pre2023':old2022,'nonself_candidates_pre2024':old2023,
   'historical_sequence_screen_status':'INCOMPLETE' if incomplete else ('CANDIDATES_FOUND' if old2023 else 'NO_HITS_AT_THRESHOLD_NOT_ABSENCE_PROOF'),
   'template_scarcity_status':'UNDETERMINED','checkpoint_training_overlap':'UNDETERMINED'})
print('SCREENING CANDIDATES (not confirmed templates):')
for r in summary_rows:print(r['target_id'],r['historical_sequence_screen_status'],r['nonself_candidates_pre2024'])
assert len(summary_rows)==10


In [ ]:
# Immutable and validated archive. This cell writes ONLY to a brand-new evaluation folder.
assert not OUTPUT.exists(),f'Output exists: {OUTPUT}. Do not overwrite existing research artifacts.'
assert not STAGE.exists(),f'Staging exists: {STAGE}. Inspect before reuse.'
STAGE.mkdir(parents=True)
def save_csv(p,rows,keys):
 with p.open('w',newline='') as h:
  w=csv.DictWriter(h,fieldnames=keys,lineterminator='\n');w.writeheader();w.writerows(rows)
save_csv(STAGE/'official_release_dates.csv',release_rows,list(release_rows[0]))
save_csv(STAGE/'rna_sequence_search_state.csv',search_state,['target_id','returned_entities','total_reported','truncated','search_status','error'])
save_csv(STAGE/'historical_sequence_screen_summary.csv',summary_rows,list(summary_rows[0]))
save_csv(STAGE/'rcsb_candidate_entities.csv',candidate_rows,
 ['target_id','candidate_entity_id','candidate_pdb_id','candidate_initial_release_date','is_self_pdb',
  'before_end_2022','before_end_2023','candidate_only_not_training_proof'])
record={'status':'complete_with_uncertainties','run_id':OUT_ID,
 'created_utc':datetime.now(timezone.utc).isoformat(),'source_evidence_id':EVIDENCE.name,
 'source_evidence_sha256':sha256(EVIDENCE/'summary.json'),
 'release_api':'https://data.rcsb.org/rest/v1/core/entry/{pdb_id}',
 'search_api':SEARCH_API,'identity_cutoff':IDENTITY_CUTOFF,'evalue_cutoff':EVALUE_CUTOFF,
 'max_results_per_target':MAX_RESULTS_PER_TARGET,'screening_cutoffs_scenarios':CUTOFFS,
 'total_targets':len(summary_rows),'returned_candidate_entities':len(candidate_rows),
 'incomplete_targets':sum(r['historical_sequence_screen_status']=='INCOMPLETE' for r in summary_rows),
 'disclaimer':'Current PDB search + historical release-date filter. No proof of true historical availability, checkpoint training exclusion, exact structure-template similarity or template scarcity.'}
(STAGE/'summary.json').write_text(json.dumps(record,indent=2)+'\n')
files=[p.name for p in STAGE.iterdir() if p.is_file()]
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,OUTPUT)
for f in files:assert sha256(OUTPUT/f)==sha256(STAGE/f)
(OUTPUT/'FINALIZED.json').write_text(json.dumps({'status':'complete','run_id':OUT_ID,
 'files':{f:sha256(OUTPUT/f) for f in files}},indent=2)+'\n')
print('OFFICIAL DATES + CANDIDATE SCREEN SAVED, VERIFIED:',OUTPUT)
print('NO GPU; NO INFERENCE; NO GITHUB WRITE')


## Interpreting results safely

- **Before cutoff** = the candidate entry's currently reported *initial release date* precedes the chosen **scenario cutoff**; not proof that model training included it.
- **After cutoff** = direct release-date evidence only; not proof of exclusion from pretraining or homology databases.
- **Candidate** = sequence-similar RNA polymer entity from the current PDB archive under the requested threshold. This is **not** measured RNA 3D template similarity or exact model input availability.
- **Zero hit** = no hit *at the selected identity / e-value search thresholds under the currently indexed archive*, not a proof of structural novelty.
- **Truncated / API error** = incomplete; no absence claims. Some search results may be numerous and metadata retrieval can take time.
- **Next scientific requirement** = identify exact checkpoint training dataset, version, and cutoff for each model; run comprehensive training-set sequence identity, and then 3D structural similarity against pre-cutoff snapshots. Only then assign template-scarcity bins.
